# Preparação de dados para Random Forest — Previsão de Consumo de Vapor

Este notebook carrega os dados de processo (arquivos `.txt` em `0_raw`) e os dados climáticos (`1_bronze/clima_dados`), faz o merge em uma tabela unificada com freqüência horária, realiza EDA e prepara features/target para treinamento de um modelo Random Forest com scikit-learn.

In [0]:
%python
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

print("Bibliotecas importadas com sucesso.")


In [0]:
%python
# Diretório base dos dados bronze
BRONZE_DIR = "/Workspace/Previsao de consumo de vapor/1_bronze/processo_dados"

# Padrões de nome a ignorar (arquivos duplicados com timestamp no nome)
SKIP_PATTERNS = [r'\d{4}-\d{2}-\d{2}']

def parse_tag_name(filename):
    """Extrai o nome da tag do nome do arquivo (sem extensão)."""
    return filename.rsplit('.csv', 1)[0]

def extract_year_from_path(filepath):
    """Extrai o ano do caminho da pasta (2024, 2025, 2026) ou retorna None."""
    parts = filepath.split(os.sep)
    for p in parts:
        if p.isdigit() and len(p) == 4:
            return int(p)
    return None

def reconstruct_timestamps(df_time_col, year):
    """Reconstrói datetime completo a partir da coluna de tempo (HH:MM:SS.S) e do ano.
    
    Detecta mudança de dia quando o tempo diminui em relação à linha anterior.
    """
    times = pd.to_timedelta(df_time_col.astype(str))
    
    # Detectar onde o tempo diminui (mudança de dia)
    day_rolls = (times.diff() < pd.Timedelta(0)).cumsum()
    
    # Construir datetime base: 01-JAN do ano + dia do ano (0-indexed) + tempo
    base_date = pd.Timestamp(year=year, month=1, day=1)
    full_ts = base_date + pd.to_timedelta(day_rolls, unit='D') + times
    
    return full_ts

def load_csv_file(filepath):
    """Lê um arquivo .csv de processo da pasta 1_bronze.
    
    Os CSVs têm colunas ts (datetime completo YYYY-MM-DD HH:MM:SS.S) e value.
    """
    df = pd.read_csv(filepath)
    
    # Garantir colunas esperadas
    if 'ts' not in df.columns or 'value' not in df.columns:
        raise ValueError(f"Colunas inesperadas: {list(df.columns)}")
    
    year = extract_year_from_path(filepath)
    if year is None:
        # SF380 ou outras pastas sem ano: usar 2024 como fallback
        year = 2024
    
    # Os CSVs já contêm datetime completo
    df['ts'] = pd.to_datetime(df['ts'])
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    df = df[['ts', 'value']].dropna(subset=['ts'])
    
    return df

def find_csv_files(base_dir):
    """Encontra recursivamente todos os arquivos .csv, ignorando duplicados com timestamp no nome.
    Retorna (arquivos_a_ler, arquivos_pulados_por_padrao, arquivos_nao_csv)."""
    csv_files = []
    skipped_by_pattern = []
    non_csv = []
    for root, dirs, files in os.walk(base_dir):
        for f in sorted(files):
            if not f.endswith('.csv'):
                non_csv.append(os.path.join(root, f))
                continue
            # Pular arquivos com timestamp no nome (duplicados)
            if any(re.search(p, f) for p in SKIP_PATTERNS):
                skipped_by_pattern.append(os.path.join(root, f))
                continue
            csv_files.append(os.path.join(root, f))
    return sorted(csv_files), sorted(skipped_by_pattern), sorted(non_csv)

# Encontrar e carregar todos os arquivos .csv do 1_bronze
csv_files, skipped_by_pattern, non_csv = find_csv_files(BRONZE_DIR)
print(f"Total de arquivos .csv encontrados: {len(csv_files)}")
print(f"Arquivos .csv pulados por SKIP_PATTERNS (timestamp no nome): {len(skipped_by_pattern)}")
if skipped_by_pattern:
    print("  Arquivos pulados:")
    for f in skipped_by_pattern[:20]:
        print(f"    {os.path.basename(f)}")
    if len(skipped_by_pattern) > 20:
        print(f"    ... e mais {len(skipped_by_pattern)-20} arquivos")
print(f"Arquivos não-.csv encontrados: {len(non_csv)}")
if non_csv:
    exts = set(os.path.splitext(f)[1] for f in non_csv)
    print(f"  Extensões: {exts}")
    for f in non_csv[:10]:
        print(f"    {os.path.basename(f)}")

# Carregar cada arquivo em um dicionário de DataFrames
all_series = {}
erros = []

for fpath in csv_files:
    tag = parse_tag_name(os.path.basename(fpath))
    try:
        df = load_csv_file(fpath)
        if len(df) > 0:
            if tag in all_series:
                all_series[tag] = pd.concat([all_series[tag], df], ignore_index=True)
            else:
                all_series[tag] = df
    except Exception as e:
        erros.append((tag, str(e)))

print(f"Tags carregadas com sucesso: {len(all_series)}")
if erros:
    print(f"Erros: {len(erros)}")
    for tag, err in erros:
        print(f"  {tag}: {err}")

# Mostrar resumo das primeiras tags
print("\n=== Resumo das tags carregadas ===")
for tag, df in list(all_series.items())[:5]:
    print(f"  {tag}: {len(df):,} registros | {df['ts'].min()} a {df['ts'].max()}")
print(f"  ... e mais {len(all_series)-5} tags")


In [0]:
%python
# Resample cada tag para freqüência horária (média) e junta em uma tabela wide
hourly_frames = []
for tag, df in all_series.items():
    s = df.set_index('ts')['value']
    s_resampled = s.resample('1h').mean()
    s_resampled.name = tag
    hourly_frames.append(s_resampled.to_frame())

# Concatenar todas as tags em um único DataFrame
df_process = pd.concat(hourly_frames, axis=1)
df_process.index.name = 'datetime'

print(f"Tabela wide de processo (horária): {df_process.shape}")
print(f"Período: {df_process.index.min()} a {df_process.index.max()}")
print(f"\nPrimeiras 3 linhas:")
display(df_process.head(3))


In [0]:
%python
# Carregar dados climáticos
CLIMA_PATH = "/Workspace/Previsao de consumo de vapor/1_bronze/clima_dados/dados_meteorologicos_sao_jose_dos_campos_20212026.csv - dados_meteorologicos_sao_jose_dos_campos_20212026.csv"

df_clima = pd.read_csv(CLIMA_PATH)
df_clima['datetime_local'] = pd.to_datetime(df_clima['datetime_local'])
df_clima = df_clima.set_index('datetime_local')
df_clima.index.name = 'datetime'

print(f"Dados climáticos: {df_clima.shape}")
print(f"Período: {df_clima.index.min()} a {df_clima.index.max()}")
print(f"Colunas: {list(df_clima.columns)}")
print()
display(df_clima.head(3))

# Merge: processo + clima (join outer para preservar todas as horas)
df_merged = df_process.join(df_clima, how='inner')

print(f"\n=== Tabela unificada (processo + clima) ===")
print(f"Shape: {df_merged.shape}")
print(f"Período: {df_merged.index.min()} a {df_merged.index.max()}")
print(f"Colunas: {list(df_merged.columns)}")
print(f"\nPrimeiras 3 linhas:")
display(df_merged.head(3))


In [0]:
%python
print("=== SHAPE ===")
print(f"Linhas: {df_merged.shape[0]:,} | Colunas: {df_merged.shape[1]}")

print("\n=== DESCRIBE ===")
display(df_merged.describe().T)

print("\n=== MISSING VALUES ===")
missing = df_merged.isnull().sum()
missing_pct = (missing / len(df_merged) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_df = missing_df[missing_df['missing_count'] > 0].sort_values('missing_pct', ascending=False)
print(f"Colunas com missing: {len(missing_df)} de {len(df_merged.columns)}")
if len(missing_df) > 0:
    display(missing_df)
else:
    print("Nenhum missing value encontrado.")


In [0]:
%python
# Tabela de tipos de features
feature_types = []
for col in df_merged.columns:
    dtype = df_merged[col].dtype
    n_unique = df_merged[col].nunique()
    if dtype in ('float64', 'float32', 'int64', 'int32'):
        if n_unique <= 20:
            ftype = 'numeric ordinal'
        else:
            ftype = 'numeric float'
    elif dtype == 'object':
        ftype = 'categorical/nominal'
    else:
        ftype = str(dtype)
    feature_types.append({'feature': col, 'type': ftype, 'n_unique': n_unique})

feature_type_df = pd.DataFrame(feature_types)
print("=== Tabela de tipos de features ===")
display(feature_type_df)


In [0]:
%python
# Mapa de correlações entre as variáveis numéricas
numeric_cols = df_merged.select_dtypes(include=[np.number]).columns
corr_matrix = df_merged[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Mapa de Correlações — Variáveis de Processo e Clima', fontsize=14)
plt.tight_layout()
plt.show()


In [0]:
# ============================================================
# CORRELAÇÃO DAS FEATURES COM CADA TARGET (4 SISTEMAS)
# ============================================================
# Calcula a correlação de Pearson de cada feature com cada
# um dos 4 targets (sistemas distintos) e exibe lado a lado.
# ============================================================

MULTI_TARGETS = ['11FI208-02', '11FI502-30.PV', '21FI550-30', '21FI551-10']

# Validar quais existem no DataFrame
multi_targets_valid = [t for t in MULTI_TARGETS if t in df_merged.columns]
print(f"Targets encontrados: {multi_targets_valid}")

# Colunas a excluir da análise (os próprios targets e seus derivados _TOT)
exclude_cols = set(multi_targets_valid)
exclude_cols.update([c for c in df_merged.columns if '_TOT' in c])

feature_cols = [c for c in df_merged.select_dtypes(include=[np.number]).columns
                if c not in exclude_cols]

# Calcular correlação de cada feature com cada target
corr_dict = {}
for tgt in multi_targets_valid:
    corr = df_merged[feature_cols].corrwith(df_merged[tgt]).dropna()
    corr_dict[tgt] = corr

# Montar tabela comparativa
multi_corr_df = pd.DataFrame(corr_dict)
multi_corr_df.index.name = 'feature'

# Adicionar coluna com média de |correlação| across targets
multi_corr_df['media_abs'] = multi_corr_df[multi_targets_valid].abs().mean(axis=1)

# Ordenar pela média de correlação absoluta (features mais relevantes no geral)
multi_corr_df = multi_corr_df.sort_values('media_abs', ascending=False)

print(f"\n=== Correlação de Pearson por Target (sistemas distintos) ===")
print(f"Features analisadas: {len(multi_corr_df)}")
print(f"Targets: {multi_targets_valid}")
print()
display(multi_corr_df.round(4))

# Resumo: features com |corr| > 0.5 em pelo menos um target
high_corr = multi_corr_df[(multi_corr_df[multi_targets_valid].abs() > 0.5).any(axis=1)]
print(f"\nFeatures com |corr| > 0.5 em pelo menos um target: {len(high_corr)}")
display(high_corr.round(4))

# Gráfico: top 15 features para cada target (subplots lado a lado)
fig, axes = plt.subplots(1, len(multi_targets_valid), figsize=(6*len(multi_targets_valid), 8), sharey=False)
if len(multi_targets_valid) == 1:
    axes = [axes]

top_n = 15
for ax, tgt in zip(axes, multi_targets_valid):
    top = multi_corr_df[tgt].abs().sort_values(ascending=False).head(top_n)
    top_features = top.index
    vals = multi_corr_df.loc[top_features, tgt].values
    colors = ['#2196F3' if v >= 0 else '#F44336' for v in vals]
    ax.barh(range(top_n), vals, color=colors, edgecolor='white')
    ax.set_yticks(range(top_n))
    ax.set_yticklabels(top_features, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel('Correlação', fontsize=10)
    ax.set_title(tgt, fontsize=11, fontweight='bold')
    ax.axvline(x=0, color='black', linewidth=0.8)

fig.suptitle(f'Top {top_n} Features por Target (Sistemas Distintos)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [0]:
%python
# Visualização de uma amostra temporal das variáveis de processo (primeiras 4 tags com dados)
fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)

tags_with_data = [c for c in df_merged.columns if df_merged[c].notna().sum() > 0][:4]

for i, tag in enumerate(tags_with_data):
    # Plotar uma amostra de 7 dias para visualização clara
    sample = df_merged[tag].dropna()
    if len(sample) > 0:
        sample_7d = sample.loc[sample.index[:24*7]]  # primeiros 7 dias
        axes[i].plot(sample_7d.index, sample_7d.values, linewidth=0.8)
        axes[i].set_ylabel(tag, fontsize=10)
        axes[i].tick_params(axis='x', rotation=30)

axes[0].set_title('Amostra Temporal (7 dias) — Primeiras 4 Tags de Processo', fontsize=13)
plt.tight_layout()
plt.show()


In [0]:
# ============================================================
# DEFINIÇÃO DO TARGET
# ============================================================
# O projeto é "Previsão de consumo de vapor".
# Tags de fluxo (FI) são candidatas a target (medem vazão de vapor).
# Ajuste a variável TARGET abaixo conforme a tag que representa
# o consumo de vapor a ser previsto.
# ============================================================

# 4 targets principais — sistemas distintos de consumo de vapor
MULTI_TARGETS = ['11FI208-02', '11FI502-30.PV', '21FI550-30', '21FI551-10']

# Validar quais existem no DataFrame
multi_targets_valid = [t for t in MULTI_TARGETS if t in df_merged.columns]
multi_targets_ausentes = [t for t in MULTI_TARGETS if t not in df_merged.columns]

print("=== Targets (sistemas distintos) ===")
for t in multi_targets_valid:
    n = df_merged[t].notna().sum()
    print(f"  {t}: {n:,} registros não-nulos")
if multi_targets_ausentes:
    print("\n--- Tags não encontradas no DataFrame ---")
    for t in multi_targets_ausentes:
        print(f"  {t}")

# Targets derivados (_TOT_DAY, _TOT) — excluir das features (vazamento de dados)
TARGET_DERIVED = [c for c in df_merged.columns if '_TOT' in c and c not in multi_targets_valid]
print(f"\nTargets derivados excluídos das features: {TARGET_DERIVED}")

# Features: todas as colunas exceto targets principais e derivados
FEATURES = [c for c in df_merged.columns
            if c not in multi_targets_valid and c not in TARGET_DERIVED]

print(f"\n>>> Targets ({len(multi_targets_valid)}): {multi_targets_valid}")
print(f">>> Total de features: {len(FEATURES)}")

In [0]:
%python
# ============================================================
# FEATURE ENGINEERING TEMPORAL
# ============================================================
# Adicionar features derivadas do timestamp (ciclo sazonal)
# ============================================================

df_prep = df_merged.copy()

# Features cíclicas (hora do dia, dia da semana, mês)
df_prep['hour'] = df_prep.index.hour
df_prep['dayofweek'] = df_prep.index.dayofweek
df_prep['month'] = df_prep.index.month
df_prep['dayofyear'] = df_prep.index.dayofyear

# Encoding senoidal/cossenoidal para preservar ciclicidade
df_prep['hour_sin'] = np.sin(2 * np.pi * df_prep['hour'] / 24)
df_prep['hour_cos'] = np.cos(2 * np.pi * df_prep['hour'] / 24)
df_prep['dow_sin'] = np.sin(2 * np.pi * df_prep['dayofweek'] / 7)
df_prep['dow_cos'] = np.cos(2 * np.pi * df_prep['dayofweek'] / 7)
df_prep['month_sin'] = np.sin(2 * np.pi * df_prep['month'] / 12)
df_prep['month_cos'] = np.cos(2 * np.pi * df_prep['month'] / 12)

# Lags e médias móveis para CADA target (4 sistemas)
for tgt in multi_targets_valid:
    for lag in [1, 3, 6, 12, 24]:
        df_prep[f'{tgt}_lag_{lag}h'] = df_prep[tgt].shift(lag)
    for window in [3, 6, 12, 24]:
        df_prep[f'{tgt}_ma_{window}h'] = df_prep[tgt].rolling(window=window, min_periods=1).mean()

print(f"DataFrame com features temporais: {df_prep.shape}")
print("Novas features:")
new_feats = ['hour', 'dayofweek', 'month', 'dayofyear',
            'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos']
print(f"  Temporais: {new_feats}")
for tgt in multi_targets_valid:
    lag_feats = [c for c in df_prep.columns if c.startswith(f'{tgt}_lag_')]
    ma_feats = [c for c in df_prep.columns if c.startswith(f'{tgt}_ma_')]
    print(f"  {tgt} — Lags: {lag_feats}")
    print(f"  {tgt} — Médias móveis: {ma_feats}")


In [0]:
%python
# ============================================================
# PREPARAÇÃO FINAL: imputação, split treino/teste (temporal)
# ============================================================
# Random Forest não exige scaling, mas precisa de valores numéricos
# sem NaN. Imputamos com a média (simplesImputer) dentro do Pipeline.
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Remover linhas onde TODOS os targets são nulos
df_model = df_prep.dropna(subset=multi_targets_valid, how='all').copy()

# Features finais: todas as colunas exceto os targets principais
FEATURES_FINAL = [c for c in df_model.columns if c not in multi_targets_valid]

X = df_model[FEATURES_FINAL]

print(f"Dados para modelagem: {X.shape}")
print(f"Targets: {multi_targets_valid}")
print(f"Features ({len(FEATURES_FINAL)})")

# Split temporal: 80% treino, 20% teste (sem embaralhar, pois é série temporal)
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]

print(f"\n=== Split Temporal ===")
print(f"Treino: {X_train.shape} | {X_train.index.min()} a {X_train.index.max()}")
print(f"Teste:  {X_test.shape} | {X_test.index.min()} a {X_test.index.max()}")

# Disponibilidade por target no treino/teste
print(f"\n--- Registros não-nulos por target ---")
for tgt in multi_targets_valid:
    n_train = df_model.iloc[:split_idx][tgt].notna().sum()
    n_test = df_model.iloc[split_idx:][tgt].notna().sum()
    print(f"  {tgt}: treino={n_train:,} | teste={n_test:,}")

# Pipeline de imputação (será reutilizado no treinamento do RF)
imputer = SimpleImputer(strategy='mean')
print(f"\nPipeline de imputação criado (SimpleImputer strategy='mean').")
print("Será incorporado no Pipeline do RandomForest no notebook de treinamento.")


In [0]:
%python
# ============================================================
# SALVAR TABELA PREPARADA PARA O RANDOM FOREST
# ============================================================
# Salvar como CSV na pasta 2_silver para uso no notebook de treinamento
# ============================================================

import io
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

SILVER_DIR = "/Previsao de consumo de vapor/2_silver"
SILVER_FILE = f"{SILVER_DIR}/rf_consumo_vapor_prepared.csv"

# Criar diretório de destino
w.workspace.mkdirs(SILVER_DIR)

# Resetar índice para incluir datetime como coluna
df_save = df_model.reset_index()

# Escrever CSV em memória e fazer upload via FUSE
csv_fuse_path = f"/Workspace{SILVER_FILE}"
df_save.to_csv(csv_fuse_path, index=False)

print(f"Tabela preparada salva em: {SILVER_FILE}")
print(f"Shape: {df_save.shape}")
print(f"Colunas: {list(df_save.columns)}")
print(f"\nAmostra:")
display(df_save.head(5))


In [0]:
# ============================================================
# FEATURE IMPORTANCE — RANDOM FOREST (4 SISTEMAS)
# ============================================================
# Treina um RF por sistema, extrai feature_importances_ e
# visualiza o Top-15 de cada um + ranking consolidado.
# ============================================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 100

# Carregar variáveis se cells 11-13 não foram executadas
try:
    _ = multi_targets_valid, df_model, FEATURES_FINAL
except NameError:
    SILVER_FILE = "/Workspace/Previsao de consumo de vapor/2_silver/rf_consumo_vapor_prepared.csv"
    df_model = pd.read_csv(SILVER_FILE)
    if 'datetime' in df_model.columns:
        df_model['datetime'] = pd.to_datetime(df_model['datetime'])
        df_model = df_model.set_index('datetime').sort_index()
    MULTI_TARGETS = ['11FI208-02', '11FI502-30.PV', '21FI550-30', '21FI551-10']
    multi_targets_valid = [t for t in MULTI_TARGETS if t in df_model.columns]
    FEATURES_FINAL = [c for c in df_model.columns if c not in multi_targets_valid]
    print(f"Dados carregados do CSV: {df_model.shape} | {len(FEATURES_FINAL)} features")

print(f"Total de features: {len(FEATURES_FINAL)}")

# --- 1. TREINAR RF POR SISTEMA E EXTRAIR IMPORTÂNCIA ---
all_importances = {}

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

for i, tgt in enumerate(multi_targets_valid):
    # Filtrar linhas onde o target é não-nulo
    mask = df_model[tgt].notna()
    X_t = df_model.loc[mask, FEATURES_FINAL].copy()
    y_t = df_model.loc[mask, tgt]

    # Split temporal 80/20
    split = int(len(X_t) * 0.8)
    X_tr, X_te = X_t.iloc[:split], X_t.iloc[split:]
    y_tr, y_te = y_t.iloc[:split], y_t.iloc[split:]

    # Dropar colunas totalmente NaN no treino (SimpleImputer as remove,
    # causando mismatch entre FEATURES_FINAL e rf.feature_importances_)
    valid_cols = X_tr.columns[X_tr.notna().any()].tolist()
    X_tr = X_tr[valid_cols]
    X_te = X_te[valid_cols]

    # Pipeline: imputação + RF
    pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('rf', RandomForestRegressor(
            n_estimators=100, max_depth=15,
            random_state=42, n_jobs=-1
        ))
    ])
    pipe.fit(X_tr, y_tr)

    # Feature importance — usar valid_cols (pode diferir de FEATURES_FINAL)
    rf = pipe.named_steps['rf']
    feat_df = pd.DataFrame({
        'feature': valid_cols,
        'importance': rf.feature_importances_
    }).sort_values('importance', ascending=False)

    all_importances[tgt] = feat_df

    # Métricas
    y_pred = pipe.predict(X_te)
    r2 = r2_score(y_te, y_pred)
    mae = mean_absolute_error(y_te, y_pred)

    # Plotar Top 15
    top_n = 15
    top = feat_df.head(top_n)
    ax = axes[i]
    ax.barh(range(top_n), top['importance'].values[::-1], color=colors[i], alpha=0.8)
    ax.set_yticks(range(top_n))
    ax.set_yticklabels(top['feature'].values[::-1], fontsize=8)
    ax.set_xlabel('Importância (Gini)')
    ax.set_title(f'{tgt}  (n={len(y_t):,} | R²={r2:.3f} | MAE={mae:.2f})', fontsize=10)
    ax.text(0.95, 0.05, f'R²={r2:.3f}\nMAE={mae:.2f}',
            transform=ax.transAxes, fontsize=9,
            verticalalignment='bottom', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Feature Importance — Random Forest (Top 15 por sistema)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# --- 2. RANKING CONSOLIDADO (média across targets) ---
print(f"\n{'='*70}")
print('TOP 15 FEATURES — IMPORTÂNCIA MÉDIA (4 SISTEMAS)')
print(f"{'='*70}")

# Normalizar cada target para soma = 1
# Usar union de features que apareceram em pelo menos um sistema
all_features = set()
for tgt in multi_targets_valid:
    all_features.update(all_importances[tgt]['feature'].tolist())
all_features = sorted(all_features)

combined = pd.DataFrame({'feature': all_features})
for tgt in multi_targets_valid:
    tmp = all_importances[tgt].rename(columns={'importance': f'imp_{tgt}'})
    combined = combined.merge(tmp[['feature', f'imp_{tgt}']], on='feature', how='left')

imp_cols = [c for c in combined.columns if c.startswith('imp_')]
combined['importance_mean'] = combined[imp_cols].mean(axis=1)
combined = combined.sort_values('importance_mean', ascending=False)

print(f"\n{'#':>3}  {'Feature':<40} {'Imp. Média':>12}")
print(f"{'-'*58}")
for j, (_, row) in enumerate(combined.head(15).iterrows(), 1):
    print(f'{j:>3}  {row["feature"]:<40} {row["importance_mean"]:>12.4f}')

# --- 3. GRÁFICO CONSOLIDADO ---
fig2, ax2 = plt.subplots(figsize=(12, 8))
top15 = combined.head(15)
ax2.barh(range(15), top15['importance_mean'].values[::-1], color='#3F51B5', alpha=0.8)
ax2.set_yticks(range(15))
ax2.set_yticklabels(top15['feature'].values[::-1], fontsize=9)
ax2.set_xlabel('Importância Média (normalizada)')
ax2.set_title('Top 15 Features — Importância Média Consolidada (4 Sistemas)',
              fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# --- 4. AGRUPAR POR CATEGORIA ---
print(f"\n{'='*70}")
print('IMPORTÂNCIA POR CATEGORIA DE FEATURE')
print(f"{'='*70}")

def categorize_feature(name):
    if '_lag_' in name:
        return 'Lags (autoregressivo)'
    elif '_ma_' in name:
        return 'Médias móveis'
    elif name in ('hour', 'hour_sin', 'hour_cos'):
        return 'Sazonalidade horária'
    elif name in ('dayofweek', 'dow_sin', 'dow_cos'):
        return 'Sazonalidade semanal'
    elif name in ('month', 'month_sin', 'month_cos', 'dayofyear'):
        return 'Sazonalidade mensal/anual'
    elif name in ('temperature_C', 'relative_humidity_pct',
                  'surface_pressure_hPa', 'pressure_msl_hPa'):
        return 'Clima'
    else:
        return 'Processo (pressão, temperatura, controle)'

combined['categoria'] = combined['feature'].apply(categorize_feature)
cat_imp = combined.groupby('categoria')['importance_mean'].agg(['sum', 'mean', 'count'])
cat_imp = cat_imp.sort_values('sum', ascending=False)

print(f"\n{'Categoria':<45} {'Soma':>10} {'Média':>10} {'Nº feats':>10}")
print(f"{'-'*78}")
for cat, row in cat_imp.iterrows():
    print(f'{cat:<45} {row["sum"]:>10.4f} {row["mean"]:>10.4f} {int(row["count"]):>10}')